In [ ]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "../input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install xgboost lightgbm --quiet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

In [ ]:
file_path = "/kaggle/input/findata/financial_transaction_dataset_with_risk.csv"
df = pd.read_csv(file_path)

In [ ]:
# ✅ Step 4: Display Dataset Info
print("Dataset Shape:", df.shape)
print("\nDataset Preview:\n", df.head())


In [ ]:
encoders = {}  # Dictionary to store label encoders for each column

for col in ["CurrencyType", "TransactionType", "TransactionStatus", "PaymentMethod"]:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col])

In [ ]:
features = ["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig", "CurrencyType",
            "TransactionType", "TransactionStatus", "PaymentMethod"]
target = "RiskLevel"

X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=0.2, random_state=42)


In [ ]:
base_models = [
    ('linear', LinearRegression()),
    ('random_forest', RandomForestRegressor(n_estimators=200, random_state=42)),
    ('xgboost', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)),
    ('lightgbm', LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42))
]


In [ ]:
stacking_model = StackingRegressor(estimators=base_models, final_estimator=GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))


In [ ]:
print("\nTraining the Stacking Model...")
stacking_model.fit(X_train, y_train)

In [ ]:
y_pred = stacking_model.predict(X_test)

# ✅ Step 11: Evaluate Model Performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n📊 Model Performance:")
print(f"MSE = {mse:.4f}")
print(f"R² Score = {r2:.4f}")


In [ ]:
joblib.dump(stacking_model, "stacking_fraud_model.pkl")

In [ ]:
# ✅ Define new transaction
new_transaction = pd.DataFrame({
    "TransactionAmount": [30000],
    "OldBalanceOrig": [80000],
    "NewBalanceOrig": [50000],
    "CurrencyType": ["USD"],  # This might be an unseen category
    "TransactionType": ["Transfer"],
    "TransactionStatus": ["Success"],
    "PaymentMethod": ["Credit Card"]
})

# ✅ Encode categorical variables safely
for col in ["CurrencyType", "TransactionType", "TransactionStatus", "PaymentMethod"]:
    if col in encoders:  # Ensure encoder exists
        new_transaction[col] = new_transaction[col].map(lambda x: encoders[col].transform([x])[0] if x in encoders[col].classes_ else -1)

# ✅ Ensure all required columns exist in new_transaction
for col in X_train.columns:
    if col not in new_transaction.columns:
        new_transaction[col] = 0  # Fill missing columns with 0

# ✅ Predict Risk Level
predicted_risk = stacking_model.predict(new_transaction)
print(f"\n🔍 Predicted Risk Level for New Transaction: {predicted_risk[0]:.4f}")


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.xlabel("Actual Risk Level")
plt.ylabel("Predicted Risk Level")
plt.title("Actual vs. Predicted Risk Levels")
plt.show()